In [0]:
from pyspark.sql.functions import col
import pyspark.sql.functions as F

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.types import *

In [0]:
bronze_flight_table = "us_project_catalog.raw.Flight_delta_table"

silver_df = spark.read.table(bronze_flight_table)

In [0]:
silver_table_name = "us_project_catalog.silver_schema.flight_silver_table"
silver_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(silver_table_name)

delta_table_silver = DeltaTable.forName(spark, silver_table_name)


In [0]:
spark.read.table(silver_table_name).show()

+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+-----------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+---------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+------------------+------------------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------------+-----------------+-------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+------------+-------------+---------------+------------------+--------------+--------------------+-----------+-----------+-----------+-------------+----------------+------------+-----------

connecting the silver delta table with snowflake using snowflake connector  


In [0]:
sfOptions = {
    "host": "FVLPSCQ-NE73500.snowflakecomputing.com",
    "sfaccount": "FVLPSCQ-NE73500",
    "sfUser": "PURUU112211",
    "sfPassword": "Puru@8764917565",
    "sfDatabase": "databrick_project_db",
    "sfSchema": "Gold_schema",
    "sfWarehouse": "compute_wh"
}

In [0]:
try_df = spark.sql(
"""
select year as flight_year,
month  as flight_month,
IATA_CODE_Reporting_Airline as airline_code , 
count(*) as total_flights , 
sum(diverted) as total_diverted ,
sum(cancelled) as total_cancelled , 
round(avg(case when lower(delay_cateory) in ('major_delay' , 'minor_delay') then 1 else 0  end),2) as Average_delay ,
round(avg(case when lower(delay_cateory) = 'on time' then 1.0 else 0.0 end )*100 ,2) as on_time_percentage ,

round(avg(case when cast(carrierdelay as DOUBLE) > 0 then 1.0 else 0.0 end) * 100, 2) as carrierdelay_percentage,
round(avg(case when cast(weatherdelay as DOUBLE) > 0 then 1.0 else 0.0 end) * 100, 2) as weatherdelay_percentage,
round(avg(case when cast(nasdelay as DOUBLE) > 0 then 1.0 else 0.0 end) * 100, 2) as nasdelay_percentage,
round(avg(case when cast(securitydelay as DOUBLE) > 0 then 1.0 else 0.0 end) * 100, 2) as securitydelay_percentage,
round(avg(case when cast(lateaircraftdelay as DOUBLE) > 0 then 1.0 else 0.0 end) * 100, 2) as lateaircraftdelay_percentage

from us_project_catalog.silver_schema.flight_silver_table 
group by year , month , IATA_CODE_Reporting_Airline;""")



# Write your DataFrame directly to Snowflake using the connector
try_df.write.format("snowflake").options(**sfOptions).option("dbtable", "flight_summary_table").mode("append").save()


In [0]:
    try_df2 = spark.sql("""
    select 
    year as flight_year, 
    CONCAT(origin,' -> ',dest)   as route, 
    COUNT(*) AS total_flights,
    AVG(DEPDELAY) AS avg_dep_delay,
    AVG(ARRDELAY) AS avg_arr_delay,
    SUM(CASE WHEN cast(CANCELLED as DOUBLE) = 1 THEN 1 ELSE 0 END) AS total_cancelled,
    AVG(DISTANCE) AS avg_distance
from us_project_catalog.silver_schema.flight_silver_table 
group by year , CONCAT(origin,' -> ',dest);""")
    
try_df2.write.format("snowflake").options(**sfOptions).option("dbtable", "route_based_insights").mode("append").save()

In [0]:

try_df3 = spark.sql("""
select 
year as flight_year, 
month  as flight_month , 
IATA_CODE_Reporting_Airline as airline_code ,

sum(case when lower(delay_cateory) in ('major_delay' , 'minor_delay') then 1 else 0  end) as Total_delay_flight ,
sum(depdelayminutes) as total_delay_minutes ,
sum(carrierdelay) as carrier_delay_minutes ,
sum(weatherdelay) as weather_delay_minutes , 
sum(nasdelay) as nas_delay_minutes ,
sum(securitydelay) as security_delay_minutes ,
sum(lateaircraftdelay) as late_aircraft_delay_minutes ,

round(avg(case when cast(carrierdelay as DOUBLE) > 0 then 1.0 else 0.0 end) * 100, 2) as carrierdelay_percentage,
round(avg(case when cast(weatherdelay as DOUBLE) > 0 then 1.0 else 0.0 end) * 100, 2) as weatherdelay_percentage,
round(avg(case when cast(nasdelay as DOUBLE) > 0 then 1.0 else 0.0 end) * 100, 2) as nasdelay_percentage,
round(avg(case when cast(securitydelay as DOUBLE) > 0 then 1.0 else 0.0 end) * 100, 2) as securitydelay_percentage,
round(avg(case when cast(lateaircraftdelay as DOUBLE) > 0 then 1.0 else 0.0 end) * 100, 2) as lateaircraftdelay_percentage,
current_timestamp() as gold_processed_ts

from us_project_catalog.silver_schema.flight_silver_table 
group by  flight_year , flight_month ,airline_code;""")

try_df3.write.format("snowflake").options(**sfOptions).option("dbtable", "delay_cause_analysis").mode("append").save()

In [0]:
try_df4 = spark.sql("""
select 
year as flight_year, 
month  as flight_month , 
concat(year,'_',month) as year_month, 
IATA_CODE_Reporting_Airline as airline_code,
origin as origin_code,
originstatename as origin_state,
origincityname as origin_city,
name as origin_airport_name, 
latitude as origin_lat ,
longitude as origin_lon ,
-- AGGREGRATE COLUMNS 
count(*) as departure , 
sum(cancelled) as cancelled_departure,
round(avg(depdelayminutes),2) as avg_dep_delay,
round(avg(case when lower(delay_cateory) = 'on time' then 1 else 0 end )*100 ,2) as dep_on_time_pct ,
avg(airtime) as avg_airtime ,
avg(DISTANCE) AS avg_route_distance,
count(distinct IATA_CODE_Reporting_Airline) as airline_operating,
count(distinct dest) as unique_destinations,
current_timestamp() as gold_processed_ts

from us_project_catalog.silver_schema.flight_silver_table
group by flight_year , flight_month , year_month , airline_code ,origin_code , origin_state , origin_city ,origin_airport_name ,origin_lat , origin_lon;""")

try_df4.write.format("snowflake").options(**sfOptions).option("dbtable", "airport_monthly_performance").mode("append").save()
